# 01 — Data Audit
## Goal
Before cleaning or modeling, we inspect the **actual CSV**. We do not decide what is wrong before seeing evidence.

### Prediction decision
- **Target:** `Time_taken (min)`
- **Prediction moment:** when the customer places the order
- **Important leakage rule:** `Time_Order_picked` is **not a model input**, because pickup happens after the order is placed.

### Questions this notebook answers
1. What is the shape of the data?
2. What are the data types?
3. Where are missing values?
4. Are there duplicates?
5. Are there suspicious ranges/categories?
6. Which columns are identifiers, target, usable features, or leakage-prone?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

file_path = "../data/Zomato Dataset.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (45584, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.000,4.200,30.328,78.046,30.398,78.116,12-02-2022,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.000,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.000,4.700,10.003,76.308,10.043,76.348,13-02-2022,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.000,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.000,4.700,18.562,73.917,18.652,74.007,04-03-2022,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.000,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.000,4.300,30.900,75.809,30.920,75.829,13-02-2022,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.000,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.000,4.700,26.464,80.373,26.594,80.503,14-02-2022,19:50,20:05,Fog,Jam,1,Snack,scooter,1.000,No,Metropolitian,41


In [2]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

audit

,dtype,missing_count,missing_pct,unique_values
Delivery_person_Ratings,float64,1908,4.190,28
Delivery_person_Age,float64,1854,4.070,22
Time_Orderd,object,1731,3.800,176
City,object,1200,2.630,3
multiple_deliveries,float64,993,2.180,4
Weather_conditions,object,616,1.350,6
Road_traffic_density,object,601,1.320,4
Festival,object,228,0.500,2
ID,object,0,0.000,45584
Type_of_vehicle,object,0,0.000,4


In [3]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# Numeric summary
df.describe(include="all").T

Duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,45584,45584,0xcdcd,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_person_ID,45584,1320,JAPRES11DEL02,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_person_Age,"43,730.000",NaN,NaN,NaN,29.567,5.815,15.000,25.000,30.000,35.000,50.000
Delivery_person_Ratings,"43,676.000",NaN,NaN,NaN,4.634,0.335,1.000,4.500,4.700,4.900,6.000
Restaurant_latitude,"45,584.000",NaN,NaN,NaN,17.018,8.186,-30.906,12.933,18.551,22.728,30.914
Restaurant_longitude,"45,584.000",NaN,NaN,NaN,70.230,22.886,-88.366,73.170,75.898,78.044,88.433
Delivery_location_latitude,"45,584.000",NaN,NaN,NaN,17.465,7.336,0.010,12.988,18.634,22.785,31.054
Delivery_location_longitude,"45,584.000",NaN,NaN,NaN,70.844,21.121,0.010,73.280,76.003,78.107,88.563
Order_Date,45584,44,15-03-2022,1192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time_Orderd,43853,176,21:55,461,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Inspect categorical values
categorical_cols = df.select_dtypes(include="object").columns

category_summary = []
for col in categorical_cols:
    category_summary.append({
        "column": col,
        "unique_count": df[col].nunique(dropna=True),
        "sample_values": df[col].dropna().astype(str).str.strip().unique()[:12].tolist()
    })

pd.DataFrame(category_summary)

,column,unique_count,sample_values
0,ID,45584,"[0xcdcd, 0xd987, 0x2784, 0xc8b6, 0xdb64, 0x3af..."
1,Delivery_person_ID,1320,"[DEHRES17DEL01, KOCRES16DEL01, PUNERES13DEL03,..."
2,Order_Date,44,"[12-02-2022, 13-02-2022, 04-03-2022, 14-02-202..."
3,Time_Orderd,176,"[21:55, 14:55, 17:30, 09:20, 19:50, 20:25, 20:..."
4,Time_Order_picked,193,"[22:10, 15:05, 17:40, 09:30, 20:05, 20:35, 15:..."
5,Weather_conditions,6,"[Fog, Stormy, Sandstorms, Windy, Cloudy, Sunny]"
6,Road_traffic_density,4,"[Jam, High, Medium, Low]"
7,Type_of_order,4,"[Snack, Meal, Drinks, Buffet]"
8,Type_of_vehicle,4,"[motorcycle, scooter, electric_scooter, bicycle]"
9,Festival,2,"[No, Yes]"


In [5]:
# Range / quality checks for important numeric fields
numeric_check_cols = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Vehicle_condition",
    "multiple_deliveries",
    "Time_taken (min)",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

df[numeric_check_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Delivery_person_Age,"43,730.000",29.567,5.815,15.000,25.000,30.000,35.000,50.000
Delivery_person_Ratings,"43,676.000",4.634,0.335,1.000,4.500,4.700,4.900,6.000
Vehicle_condition,"45,584.000",1.023,0.839,0.000,0.000,1.000,2.000,3.000
multiple_deliveries,"44,591.000",0.745,0.573,0.000,0.000,1.000,1.000,3.000
Time_taken (min),"45,584.000",26.294,9.384,10.000,19.000,26.000,32.000,54.000
Restaurant_latitude,"45,584.000",17.018,8.186,-30.906,12.933,18.551,22.728,30.914
Restaurant_longitude,"45,584.000",70.230,22.886,-88.366,73.170,75.898,78.044,88.433
Delivery_location_latitude,"45,584.000",17.465,7.336,0.010,12.988,18.634,22.785,31.054
Delivery_location_longitude,"45,584.000",70.844,21.121,0.010,73.280,76.003,78.107,88.563


In [6]:
# Explicit anomaly checks
checks = pd.Series({
    "ratings_below_1": (df["Delivery_person_Ratings"] < 1).sum(),
    "ratings_above_5": (df["Delivery_person_Ratings"] > 5).sum(),
    "age_below_18": (df["Delivery_person_Age"] < 18).sum(),
    "age_above_70": (df["Delivery_person_Age"] > 70).sum(),
    "vehicle_condition_outside_0_3": (~df["Vehicle_condition"].between(0, 3)).sum(),
    "multiple_deliveries_outside_0_3": (~df["multiple_deliveries"].between(0, 3)).sum(),
    "target_below_0": (df["Time_taken (min)"] <= 0).sum(),
    "restaurant_lat_zero": (df["Restaurant_latitude"] == 0).sum(),
    "restaurant_lon_zero": (df["Restaurant_longitude"] == 0).sum(),
    "negative_restaurant_lat": (df["Restaurant_latitude"] < 0).sum(),
    "negative_restaurant_lon": (df["Restaurant_longitude"] < 0).sum(),
})
checks.to_frame("count")

,count
ratings_below_1,0
ratings_above_5,53
age_below_18,38
age_above_70,0
vehicle_condition_outside_0_3,0
multiple_deliveries_outside_0_3,993
target_below_0,0
restaurant_lat_zero,3640
restaurant_lon_zero,3640
negative_restaurant_lat,431


## Initial modeling decision

We will **not** use:
- `ID` — identifier
- `Delivery_person_ID` — identifier
- `Time_taken (min)` — target
- `Time_Order_picked` — leakage / unavailable at prediction time

We will investigate the remaining columns and engineer features from information available at order placement.

### Evidence note

The GitHub projects we reviewed use this same 45,584 × 20 delivery dataset and also identify `Time_taken (min)` as the prediction target. One reference project explicitly emphasizes leakage prevention and regularized models. Another discusses GPS quality and time parsing.

These repositories are **references**, not ground truth. Our cleaning decisions are based on the actual uploaded CSV.